In [13]:
import sys
import os
import pandas as pd
import numpy as np

In [14]:
if 'google.colab' in sys.modules: 
    if not os.path.exists('/content/nlp_uni'):
        !git clone -b lab-02 https://github.com/Danylo-NULP/nlp_uni.git
    
    %cd /content/nlp_uni
    !pip install pandas -q
    sys.path.append('/content/nlp_uni')
    
    FOLDER_ID = '1LhS2rA8VAQVd_lzUwMXuHav6fSVcGO0D'
    
    os.makedirs('/content/nlp_uni/data', exist_ok=True)
    !gdown --folder https://drive.google.com/drive/folders/{FOLDER_ID} -O /content/nlp_uni/data/
    
    data_dir = '/content/nlp_uni/data'

else:
    sys.path.append(os.path.abspath('..'))
    data_dir = '../data'

In [15]:
df_raw = f'{data_dir}/raw/raw.csv'

df = pd.read_csv(df_raw)
print(f"Кількість рядків: {len(df)}")

Кількість рядків: 1500


In [16]:
from src.preprocess import preprocess

# Запуск пайплайну 
processed_premise = df['premise'].apply(preprocess)
df['premise_clean'] = processed_premise.apply(lambda x: x['clean'])
df['premise_sents'] = processed_premise.apply(lambda x: x['sentences'])

# Обробляємо Hypothesis
processed_hypo = df['hypothesis'].apply(preprocess)
df['hypothesis_clean'] = processed_hypo.apply(lambda x: x['clean'])
df['hypothesis_sents'] = processed_hypo.apply(lambda x: x['sentences'])

# Приклади 'До / Після'
print("\n15 прикладів (Premise): Raw -> Processed")
display(df[['premise', 'premise_clean']].head(15))


15 прикладів (Premise): Raw -> Processed


,premise,premise_clean
0,A woman and a young girl smiling for the camer...,A woman and a young girl smiling for the camer...
1,"A young boy wearing a gray sweater, blue jeans...","A young boy wearing a gray sweater, blue jeans..."
2,A woman with a very large black wig and giant ...,A woman with a very large black wig and giant ...
3,Oriental man dressed in tank top and shorts is...,Oriental man dressed in tank top and shorts is...
4,3 girls and one boy playing in the street.,3 girls and one boy playing in the street.
5,Two girls are going for a swim in a mountain l...,Two girls are going for a swim in a mountain l...
6,A woman fiddles with her phone at a diner.,A woman fiddles with her phone at a diner.
7,Two women are walking casually down the street...,Two women are walking casually down the street...
8,A bunch of kids in canoes on a river.,A bunch of kids in canoes on a river.
9,A white dog and two black dogs playing,A white dog and two black dogs playing


In [17]:
# 4. Статистика 'До / Після'
# 4.1 Точні дублікати
dups_raw = df.duplicated(subset=['premise', 'hypothesis']).sum()
dups_clean = df.duplicated(subset=['premise_clean', 'hypothesis_clean']).sum()
print(f"\nТочні дублі (пар речень): Raw = {dups_raw}, Processed = {dups_clean}")

# 4.2 Порожні / дуже короткі (< 5 слів)
def is_short(text):
    return len(str(text).split()) < 5

short_premise_raw = df['premise'].apply(is_short).sum()
short_premise_clean = df['premise_clean'].apply(is_short).sum()
print(f"Дуже короткі (< 5 слів) Premise: Raw = {short_premise_raw}, Processed = {short_premise_clean}")

# 4.3 Розподіл довжин (Символи)
print("\nМедіанна довжина (Символи):")
print(f"Premise:    Raw = {df['premise'].str.len().median()}, Processed = {df['premise_clean'].str.len().median()}")
print(f"Hypothesis: Raw = {df['hypothesis'].str.len().median()}, Processed = {df['hypothesis_clean'].str.len().median()}")

# 4.4 Кількість замін PII
# Рахуємо, скільки разів теги з'являються у фінальному тексті
tags = ['<URL>', '<EMAIL>', '<PHONE>']
mask_counts = {tag: 0 for tag in tags}

for text in df['premise_clean'].tolist() + df['hypothesis_clean'].tolist():
    for tag in tags:
        mask_counts[tag] += text.count(tag)

print("\nЗроблено маскувань (PII):")
for tag, count in mask_counts.items():
    print(f"{tag}: {count}")

# 5. Збереження результатів
import os
os.makedirs(f'{data_dir}/processed_v2', exist_ok=True)
os.makedirs(f'{data_dir}/sample_v2', exist_ok=True)

# Зберігаємо повний файл
cols_to_save = ['label', 'premise_clean', 'hypothesis_clean', 'premise_sents', 'hypothesis_sents']
df[cols_to_save].to_csv(f'{data_dir}/processed_v2/processed_v2.csv', index=False)

# Зберігаємо sample (перші 100 рядків) для GitHub
df[cols_to_save].head(100).to_csv(f'{data_dir}/sample_v2/sample_v2.csv', index=False)

print(f"\nДані успішно збережено у {data_dir}/processed_v2/ та {data_dir}/sample_v2/")


Точні дублі (пар речень): Raw = 0, Processed = 0
Дуже короткі (< 5 слів) Premise: Raw = 15, Processed = 15

Медіанна довжина (Символи):
Premise:    Raw = 60.0, Processed = 60.0
Hypothesis: Raw = 34.0, Processed = 34.0

Зроблено маскувань (PII):
<URL>: 0
<EMAIL>: 0
<PHONE>: 0

Дані успішно збережено у ../data/processed_v2/ та ../data/sample_v2/


In [18]:
import json
import sys
import pandas as pd

if 'google.colab' in sys.modules: 
    tests_path = '/content/nlp_uni/tests/edge_cases.jsonl'
else:
    tests_path = '../tests/edge_cases.jsonl'

# 1. Завантажуємо Edge Cases
edge_cases = []
with open(tests_path, 'r', encoding='utf-8') as f:
    for line in f:
        edge_cases.append(json.loads(line))

# 2. Проганяємо через наш пайплайн
results = []
for case in edge_cases:
    res = preprocess(case['raw_text'])
    results.append({
        "id": case['id'],
        "raw_text": case['raw_text'],
        "processed_text": res['clean'],
        "sentences": res['sentences'],
        "expected_behavior": case['expected_behavior']
    })

df_edges = pd.DataFrame(results)

# 3. Виводимо 10 найцікавіших 
interesting_ids = [3, 4, 6, 8, 9, 11, 12, 13, 15, 19]
print("=== Топ-10 Edge Cases ===")
display(df_edges[df_edges['id'].isin(interesting_ids)][['raw_text', 'processed_text', 'sentences', 'expected_behavior']])

=== Топ-10 Edge Cases ===


,raw_text,processed_text,sentences,expected_behavior
2,It’s a boy`s dog´s toy.,It's a boy's dog's toy.,[It's a boy's dog's toy.],всі типи неформатних апострофів нормалізуються...
3,Run—don't walk–quickly.,Run-don't walk-quickly.,[Run-don't walk-quickly.],довге (em) та коротке (en) тире стають дефісом -
5,Check http://example.com and www.test.org!,Check <URL> and <URL>,[Check <URL> and <URL>],два посилання замінюються на <URL>
7,Call +1-800-555-1234 or (555) 123-4567.,Call <PHONE> or (555) 123-4567.,[Call <PHONE> or (555) 123-4567.],обидва формати телефонів замінюються на <PHONE>
8,Mr. Smith is happy. He walks.,Mr. Smith is happy. He walks.,"[Mr. Smith is happy., He walks.]","2 речення, не розбивати після 'Mr.'"
10,They live in the U.S. and like it.,They live in the U.S. and like it.,[They live in the U.S. and like it.],"1 речення, не розбивати всередині та після 'U.S.'"
11,It costs $3.14 or so. It's cheap.,It costs $3.14 or so. It's cheap.,"[It costs $3.14 or so., It's cheap.]","2 речення, не розбивати на десятковому дробі 3.14"
12,Contact mr.smith@email.com at www.site.com!,Contact <EMAIL> at <URL>,[Contact <EMAIL> at <URL>],одночасна заміна на <EMAIL> та <URL>
14,"""Hello there,"" he said. ""How are you?""","""Hello there,"" he said. ""How are you?""","[""Hello there,"" he said., ""How are you?""]",нормалізація лапок і коректне розбиття на 2 ре...
18,Dr. A’s site (www.a.com) is live! Call 555-123...,Dr. A's site (<URL> is live! Call 555-123-4567.,"[Dr. A's site (<URL> is live!, Call 555-123-45...",комбо: апостроф + URL + Телефон + 2 речення (б...


In [19]:
import re

print("=== Mini-Regression Tests ===\n")

# TEST 1: Idempotence
idempotence_passed = True
failed_idempotence = []

for case in edge_cases:
    raw = case['raw_text']
    first_run = preprocess(raw)['clean']
    second_run = preprocess(first_run)['clean']
    
    if first_run != second_run:
        idempotence_passed = False
        failed_idempotence.append((raw, first_run, second_run))

if idempotence_passed:
    print("Test 1: PASSED. Пайплайн стабільний.")
else:
    print(f"Test 1: FAILED на {len(failed_idempotence)} кейсах.")
    print("Приклад фейлу:", failed_idempotence[0])

# TEST 2: No empty explosions
empty_explosions_passed = True
explosions = []

for idx, row in df.iterrows():
    raw_p = str(row['premise'])
    clean_p = row['premise_clean']
    if re.search(r'[a-zA-Z]', raw_p) and len(clean_p) == 0:
        empty_explosions_passed = False
        explosions.append(raw_p)

if empty_explosions_passed:
    print("Test 2: PASSED. Дані не зникають безслідно.")
else:
    print(f"Test 2: FAILED. Знайдено {len(explosions)} вибухів.")
    print("Приклад тексту, що зник:", explosions[0])

=== Mini-Regression Tests ===

Test 1: PASSED. Пайплайн стабільний.
Test 2: PASSED. Дані не зникають безслідно.
